# Scratch ViT BBox-Aware Interpretability Visualization

This notebook visualizes interpretability heatmaps for the custom scratch ViT baseline using **only bbox-annotated NIH samples from the selected split**.

It supports:
- single-layer CLS attention heatmaps
- attention rollout heatmaps
- bbox overlays and simple localization metrics

It does **not** use Grad-CAM and does **not** retrain the model.


In [ ]:
from pathlib import Path
import sys
import random
from typing import Optional, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display
from torch.utils.data import DataLoader, Subset

REPO_ROOT = Path("..").resolve()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from train import load_config, merge_dicts, resolve_device
from models import build_model
from data import build_nih_data_module
from interpretability import build_cls_attention_heatmap

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams["figure.figsize"] = (16, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["figure.dpi"] = 120


In [ ]:
BASE_CONFIG = REPO_ROOT / "configs/vit_baseline.yaml"
CHECKPOINT_PATH = REPO_ROOT / "artifacts/experiments/final_vit_baseline_fixed/checkpoints/vit_best_auc.pt"
# CHECKPOINT_PATH = REPO_ROOT / "artifacts/experiments/final_vit_baseline_fixed_smoke/checkpoints/vit_best_auc.pt"

SPLIT = "val"  # "val" or "test"
SELECTED_LABELS = None  # e.g. ["Cardiomegaly", "Pneumonia", "Atelectasis"]
MAX_SAMPLES_PER_LABEL = 3
BBOX_SUBSET_BATCH_SIZE = 4
NUM_WORKERS = 0

LAYER_INDEX = -1
HEAD_REDUCTION = "mean"  # "mean" or "max"
ROLLOUT_HEAD_REDUCTION = "mean"
ROLLOUT_DISCARD_RATIO = 0.0
HEATMAP_PERCENTILE = 80.0
TOP_K_PREDICTIONS = 5

OUTPUT_ROOT = REPO_ROOT / "artifacts/interpretability"
FIGURE_OUTPUT_DIR = OUTPUT_ROOT / "vit_bbox_attention_examples"
SELECTED_SAMPLES_CSV = OUTPUT_ROOT / "selected_bbox_samples.csv"
METRICS_CSV = OUTPUT_ROOT / "vit_bbox_attention_metrics.csv"


In [ ]:
if not BASE_CONFIG.exists():
    raise FileNotFoundError(f"Base config not found: {BASE_CONFIG}")
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

base_config = load_config(BASE_CONFIG)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
checkpoint_config = checkpoint.get("config", {})
config = merge_dicts(base_config, checkpoint_config)
device = resolve_device(config)

print("Device:", device)
print("Base config:", BASE_CONFIG)
print("Checkpoint:", CHECKPOINT_PATH)


In [ ]:
data_module = build_nih_data_module(config)
available_splits = list(data_module["dataloaders"].keys())
if SPLIT not in available_splits:
    raise ValueError(f"Requested split={SPLIT!r}, but available splits are {available_splits}")

split_loader = data_module["dataloaders"][SPLIT]
split_dataset = split_loader.dataset
split_frame = split_dataset.frame.reset_index(drop=True).copy()
label_names = list(data_module["labels"])

data_config = config.get("data", {})
model_config = config.get("model", {})
image_size = int(data_config.get("image_size", 224))
patch_size = int(model_config.get("patch_size", 16))
num_channels = int(data_config.get("num_channels", 1))
mean = tuple(float(v) for v in data_config.get("normalize_mean", [0.5]))
std = tuple(float(v) for v in data_config.get("normalize_std", [0.25]))

print("Selected split:", SPLIT)
print("Split size:", len(split_dataset))
print("Available splits:", available_splits)
print("Labels:", label_names)
print("Image size:", image_size)
print("Patch size:", patch_size)
print("Num channels:", num_channels)


In [ ]:
def load_bbox_annotations(annotations_dir: Path) -> pd.DataFrame | None:
    annotations_dir = Path(annotations_dir)
    bbox_path = None
    for candidate in ("BBox_List_2017.csv", "BBox_list_2017.csv"):
        candidate_path = annotations_dir / candidate
        if candidate_path.exists():
            bbox_path = candidate_path
            break

    if bbox_path is None:
        return None

    bbox_df = pd.read_csv(bbox_path)
    rename_map = {}
    for column in bbox_df.columns:
        normalized = " ".join(
            column.strip().lower().replace("[", " ").replace("]", " ").replace("_", " ").split()
        )
        compact = normalized.replace(" ", "")
        if normalized == "image index":
            rename_map[column] = "image_name"
        elif normalized == "finding label":
            rename_map[column] = "label"
        elif compact in {"bboxx", "x"}:
            rename_map[column] = "x"
        elif compact in {"bboxy", "y"}:
            rename_map[column] = "y"
        elif compact in {"bboxw", "w"}:
            rename_map[column] = "w"
        elif compact in {"bboxh", "h"}:
            rename_map[column] = "h"

    bbox_df = bbox_df.rename(columns=rename_map)
    required_columns = {"image_name", "label", "x", "y", "w", "h"}
    if not required_columns.issubset(bbox_df.columns):
        raise ValueError(
            "Bounding-box CSV is missing required columns. "
            f"Required={sorted(required_columns)} Found={list(bbox_df.columns)}"
        )

    bbox_df["image_name"] = bbox_df["image_name"].astype(str)
    bbox_df["label"] = bbox_df["label"].astype(str)
    return bbox_df


bbox_df = load_bbox_annotations(Path(data_config["annotations_dir"]))
if bbox_df is None:
    print("No NIH bbox CSV found under annotations_dir. Check BBox_List_2017.csv.")
else:
    print("Loaded bbox rows:", len(bbox_df))
    print("Unique bbox images:", bbox_df["image_name"].nunique())
    print("BBox labels:", sorted(bbox_df["label"].astype(str).unique().tolist()))


In [ ]:
def find_bbox_samples_for_split(
    dataset_or_manifest,
    bbox_df: pd.DataFrame,
    label_names: Sequence[str] | None = None,
    selected_labels: Sequence[str] | None = None,
    max_samples_per_label: int | None = None,
) -> pd.DataFrame:
    if bbox_df is None or bbox_df.empty:
        return pd.DataFrame()

    if hasattr(dataset_or_manifest, "frame"):
        split_manifest = dataset_or_manifest.frame.copy()
    elif isinstance(dataset_or_manifest, pd.DataFrame):
        split_manifest = dataset_or_manifest.copy()
    else:
        split_manifest = pd.read_csv(dataset_or_manifest)

    split_manifest = split_manifest.reset_index(drop=True)
    if "image_name" not in split_manifest.columns:
        raise ValueError("Split manifest must contain an 'image_name' column.")

    split_manifest = split_manifest.copy()
    split_manifest["dataset_index"] = split_manifest.index
    merged = split_manifest.merge(bbox_df, on="image_name", how="inner", suffixes=("", "_bbox"))

    if label_names is not None:
        allowed_labels = {str(label) for label in label_names}
        merged = merged[merged["label"].astype(str).isin(allowed_labels)]

    if selected_labels is not None:
        selected_set = {str(label) for label in selected_labels}
        merged = merged[merged["label"].astype(str).isin(selected_set)]

    merged = merged.sort_values(
        ["label", "image_name", "dataset_index", "x", "y", "w", "h"],
        kind="stable",
    ).reset_index(drop=True)

    if max_samples_per_label is not None:
        chosen_indices = []
        for label, group in merged.groupby("label", sort=True):
            unique_images = group.drop_duplicates(subset=["dataset_index"], keep="first")
            chosen_indices.extend(unique_images.head(max_samples_per_label)["dataset_index"].astype(int).tolist())
        chosen_indices = list(dict.fromkeys(chosen_indices))
        merged = merged[merged["dataset_index"].astype(int).isin(chosen_indices)].copy()
        order_map = {dataset_index: idx for idx, dataset_index in enumerate(chosen_indices)}
        merged["_selection_order"] = merged["dataset_index"].astype(int).map(order_map)
        merged = merged.sort_values(["_selection_order", "label", "x", "y", "w", "h"], kind="stable")
        merged = merged.drop(columns=["_selection_order"])

    return merged.reset_index(drop=True)


def build_bbox_subset_loader(
    dataset,
    selected_bbox_samples: pd.DataFrame,
    batch_size: int,
    num_workers: int = 0,
) -> tuple[DataLoader, pd.DataFrame, list[int]]:
    if selected_bbox_samples.empty:
        raise ValueError("selected_bbox_samples is empty.")

    ordered_indices = list(dict.fromkeys(selected_bbox_samples["dataset_index"].astype(int).tolist()))
    subset = Subset(dataset, ordered_indices)
    subset_loader = DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=num_workers > 0,
    )
    selected_images = (
        selected_bbox_samples.drop_duplicates(subset=["dataset_index"], keep="first")
        .copy()
        .reset_index(drop=True)
    )
    selected_images["subset_position"] = np.arange(len(selected_images))
    return subset_loader, selected_images, ordered_indices


requested_labels = list(SELECTED_LABELS) if SELECTED_LABELS else None
if bbox_df is None:
    selected_bbox_samples = pd.DataFrame()
else:
    selected_bbox_samples = find_bbox_samples_for_split(
        split_dataset,
        bbox_df,
        label_names=label_names,
        selected_labels=requested_labels,
        max_samples_per_label=MAX_SAMPLES_PER_LABEL,
    )

num_bbox_rows_loaded = 0 if bbox_df is None else len(bbox_df)
num_split_samples = len(split_frame)
matching_image_names = 0
available_bbox_labels = []
if bbox_df is not None:
    available_bbox_labels = sorted(bbox_df["label"].astype(str).unique().tolist())
    matching_image_names = len(set(split_frame["image_name"].astype(str)) & set(bbox_df["image_name"].astype(str)))

print("BBox rows loaded:", num_bbox_rows_loaded)
print("Split samples:", num_split_samples)
print("Matching image names between split and bbox CSV:", matching_image_names)
print("Available bbox labels:", available_bbox_labels)
print("Requested labels:", requested_labels)
print("Selected bbox rows:", len(selected_bbox_samples))

if bbox_df is not None and selected_bbox_samples.empty:
    if requested_labels:
        print(
            f"No bbox samples found for selected_labels={requested_labels}. "
            "Try SELECTED_LABELS=None or increase MAX_SAMPLES_PER_LABEL."
        )
    else:
        print(
            f"BBox CSV loaded, but no bbox-annotated images were found in split={SPLIT!r}. "
            "Try SPLIT='test' if available."
        )
else:
    display(selected_bbox_samples.head())


In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if selected_bbox_samples.empty:
    bbox_subset_loader = None
    selected_bbox_images = pd.DataFrame()
    selected_dataset_indices = []
    print("BBox subset loader was not created because no bbox samples were selected.")
else:
    bbox_subset_loader, selected_bbox_images, selected_dataset_indices = build_bbox_subset_loader(
        split_dataset,
        selected_bbox_samples,
        batch_size=BBOX_SUBSET_BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )
    selected_bbox_samples.to_csv(SELECTED_SAMPLES_CSV, index=False)
    label_counts = (
        selected_bbox_samples[["dataset_index", "label"]]
        .drop_duplicates()
        .groupby("label")
        .size()
        .sort_index()
    )
    print("BBox subset images:", len(selected_bbox_images))
    print("BBox subset batches:", len(bbox_subset_loader))
    print("Per-label selected image counts:")
    print(label_counts)
    print("Saved selected bbox samples CSV to:", SELECTED_SAMPLES_CSV)


In [ ]:
model = build_model(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded model:", model.__class__.__name__)
print("Checkpoint keys:", sorted(checkpoint.keys()))


In [ ]:
def denormalize_image(image_tensor: torch.Tensor, mean: Sequence[float], std: Sequence[float]) -> np.ndarray:
    image = image_tensor.detach().cpu().clone().float()
    for channel_index, (channel_mean, channel_std) in enumerate(zip(mean, std)):
        image[channel_index] = image[channel_index] * channel_std + channel_mean
    image = image.clamp(0.0, 1.0)
    if image.size(0) == 1:
        image = image.repeat(3, 1, 1)
    return image.permute(1, 2, 0).numpy()


def overlay_heatmap(image_array: np.ndarray, heatmap: torch.Tensor, alpha: float = 0.35) -> np.ndarray:
    base = np.clip(image_array, 0.0, 1.0)
    heatmap_array = heatmap.detach().cpu().squeeze().numpy()
    colored = plt.cm.jet(heatmap_array)[..., :3]
    return np.clip((1.0 - alpha) * base + alpha * colored, 0.0, 1.0)


def get_top_labels(logit_tensor: torch.Tensor, label_names: Sequence[str], k: int = 5) -> list[tuple[str, float]]:
    probabilities = torch.sigmoid(logit_tensor.detach().cpu())
    top_scores, top_indices = torch.topk(probabilities, k=min(k, len(label_names)))
    return [(label_names[idx], float(score)) for score, idx in zip(top_scores.tolist(), top_indices.tolist())]


def resolve_layer_index(layer_index: int | str, num_layers: int) -> int:
    if isinstance(layer_index, str):
        if layer_index == "middle":
            return num_layers // 2
        return int(layer_index)
    return int(layer_index)


def compute_attention_rollout(
    attn_maps: list[torch.Tensor],
    head_reduction: str = "mean",
    discard_ratio: float = 0.0,
) -> torch.Tensor:
    if not attn_maps:
        raise ValueError("attn_maps is empty. Run the model with return_attention=True first.")

    rollout = None
    for layer_attention in attn_maps:
        if layer_attention.ndim != 4:
            raise ValueError(
                f"Expected attention map with shape (B, heads, seq_len, seq_len), got {tuple(layer_attention.shape)}"
            )
        if head_reduction == "mean":
            fused = layer_attention.mean(dim=1)
        elif head_reduction == "max":
            fused = layer_attention.max(dim=1).values
        else:
            raise ValueError("head_reduction must be 'mean' or 'max'.")

        fused = fused.clone()
        if discard_ratio > 0:
            cls_column = fused[:, :, 0].clone()
            flattened = fused.reshape(fused.size(0), -1)
            num_discard = int(flattened.size(1) * discard_ratio)
            if num_discard > 0:
                threshold = torch.topk(flattened, k=num_discard, dim=1, largest=False).values[:, -1]
                mask = fused <= threshold.view(-1, 1, 1)
                fused[mask] = 0.0
                fused[:, :, 0] = cls_column

        identity = torch.eye(fused.size(-1), device=fused.device).unsqueeze(0)
        fused = fused + identity
        fused = fused / fused.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        rollout = fused if rollout is None else torch.bmm(fused, rollout)

    return rollout


def rollout_to_heatmap(
    attn_maps: list[torch.Tensor],
    image_size: int,
    patch_size: int,
    head_reduction: str = "mean",
    discard_ratio: float = 0.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    if image_size % patch_size != 0:
        raise ValueError("image_size must be divisible by patch_size.")

    rollout = compute_attention_rollout(
        attn_maps,
        head_reduction=head_reduction,
        discard_ratio=discard_ratio,
    )
    batch_size, seq_len, _ = rollout.shape
    grid_size = image_size // patch_size
    expected_patches = grid_size * grid_size
    if seq_len - 1 != expected_patches:
        raise ValueError(
            f"Expected {expected_patches} patches from seq_len={seq_len}, image_size={image_size}, patch_size={patch_size}."
        )

    cls_to_patches = rollout[:, 0, 1:]
    patch_grid = cls_to_patches.reshape(batch_size, grid_size, grid_size)
    heatmap = F.interpolate(
        patch_grid.unsqueeze(1),
        size=(image_size, image_size),
        mode="bilinear",
        align_corners=False,
    )
    heatmap_min = heatmap.amin(dim=(2, 3), keepdim=True)
    heatmap_max = heatmap.amax(dim=(2, 3), keepdim=True)
    heatmap = (heatmap - heatmap_min) / (heatmap_max - heatmap_min).clamp_min(1e-8)
    return patch_grid, heatmap


def draw_bboxes_on_axis(
    ax,
    boxes: pd.DataFrame,
    image_size: int,
    original_width: float = 1024,
    original_height: float = 1024,
):
    if boxes is None or boxes.empty:
        return

    scale_x = image_size / max(float(original_width), 1.0)
    scale_y = image_size / max(float(original_height), 1.0)
    for _, box in boxes.iterrows():
        x = float(box["x"]) * scale_x
        y = float(box["y"]) * scale_y
        w = float(box["w"]) * scale_x
        h = float(box["h"]) * scale_y
        ax.add_patch(Rectangle((x, y), w, h, linewidth=1.5, edgecolor="cyan", facecolor="none"))


def compute_bbox_localization_metrics(
    heatmap: torch.Tensor,
    boxes: pd.DataFrame,
    image_size: int,
    original_width: float = 1024,
    original_height: float = 1024,
    heatmap_percentile: float = 80.0,
) -> dict[str, float]:
    if boxes is None or boxes.empty:
        return {
            "attention_bbox_overlap": float("nan"),
            "attention_bbox_iou": float("nan"),
            "heatmap_mass_inside_bbox": float("nan"),
        }

    heatmap_array = heatmap.detach().cpu().squeeze().numpy()
    threshold_value = float(np.percentile(heatmap_array, heatmap_percentile))
    attention_mask = heatmap_array >= threshold_value

    bbox_mask = np.zeros((image_size, image_size), dtype=bool)
    scale_x = image_size / max(float(original_width), 1.0)
    scale_y = image_size / max(float(original_height), 1.0)
    for _, box in boxes.iterrows():
        x0 = max(0, int(np.floor(float(box["x"]) * scale_x)))
        y0 = max(0, int(np.floor(float(box["y"]) * scale_y)))
        x1 = min(image_size, int(np.ceil((float(box["x"]) + float(box["w"])) * scale_x)))
        y1 = min(image_size, int(np.ceil((float(box["y"]) + float(box["h"])) * scale_y)))
        if x1 > x0 and y1 > y0:
            bbox_mask[y0:y1, x0:x1] = True

    intersection = np.logical_and(attention_mask, bbox_mask).sum()
    union = np.logical_or(attention_mask, bbox_mask).sum()
    bbox_area = bbox_mask.sum()
    total_mass = float(heatmap_array.sum())
    mass_inside_bbox = float(heatmap_array[bbox_mask].sum()) if bbox_mask.any() else float("nan")
    return {
        "attention_bbox_overlap": float(intersection / bbox_area) if bbox_area > 0 else float("nan"),
        "attention_bbox_iou": float(intersection / union) if union > 0 else float("nan"),
        "heatmap_mass_inside_bbox": float(mass_inside_bbox / total_mass) if total_mass > 0 else float("nan"),
    }


def generate_bbox_attention_records(
    model,
    bbox_subset_loader: DataLoader,
    selected_bbox_samples: pd.DataFrame,
    selected_bbox_images: pd.DataFrame,
    label_names: Sequence[str],
    device: torch.device,
    image_size: int,
    patch_size: int,
    mean: Sequence[float],
    std: Sequence[float],
    layer_index: int | str,
    head_reduction: str,
    rollout_head_reduction: str,
    rollout_discard_ratio: float,
    heatmap_percentile: float,
    top_k: int = 5,
) -> list[dict[str, object]]:
    if bbox_subset_loader is None:
        return []

    records: list[dict[str, object]] = []
    layer_index_resolved = None
    sample_offset = 0

    for batch_images, batch_labels in bbox_subset_loader:
        batch_size = batch_images.size(0)
        batch_meta = selected_bbox_images.iloc[sample_offset : sample_offset + batch_size].reset_index(drop=True)
        sample_offset += batch_size

        batch_images_device = batch_images.to(device)
        with torch.inference_mode():
            batch_logits, batch_attn_maps = model(batch_images_device, return_attention=True)

        if layer_index_resolved is None:
            layer_index_resolved = resolve_layer_index(layer_index, len(batch_attn_maps))

        batch_logits_cpu = batch_logits.detach().cpu()
        batch_attn_maps_cpu = [attention.detach().cpu() for attention in batch_attn_maps]
        _, single_layer_heatmaps = build_cls_attention_heatmap(
            batch_attn_maps_cpu,
            image_size=image_size,
            patch_size=patch_size,
            layer_index=layer_index_resolved,
            head_reduction=head_reduction,
        )
        _, rollout_heatmaps = rollout_to_heatmap(
            batch_attn_maps_cpu,
            image_size=image_size,
            patch_size=patch_size,
            head_reduction=rollout_head_reduction,
            discard_ratio=rollout_discard_ratio,
        )

        for local_index in range(batch_size):
            metadata = batch_meta.iloc[local_index].to_dict()
            dataset_index = int(metadata["dataset_index"])
            boxes = selected_bbox_samples[selected_bbox_samples["dataset_index"].astype(int) == dataset_index].reset_index(drop=True)
            image_name = str(metadata["image_name"])
            original_width = float(metadata.get("original_width", 1024) or 1024)
            original_height = float(metadata.get("original_height", 1024) or 1024)
            image_array = denormalize_image(batch_images[local_index], mean, std)
            single_heatmap = single_layer_heatmaps[local_index]
            rollout_heatmap = rollout_heatmaps[local_index]
            single_overlay = overlay_heatmap(image_array, single_heatmap)
            rollout_overlay = overlay_heatmap(image_array, rollout_heatmap)
            true_labels = [label_names[idx] for idx, value in enumerate(batch_labels[local_index].tolist()) if value > 0.5]
            top_predictions = get_top_labels(batch_logits_cpu[local_index], label_names, k=top_k)
            bbox_labels = sorted(set(boxes["label"].astype(str).tolist()))

            single_metrics = compute_bbox_localization_metrics(
                single_heatmap,
                boxes,
                image_size=image_size,
                original_width=original_width,
                original_height=original_height,
                heatmap_percentile=heatmap_percentile,
            )
            rollout_metrics = compute_bbox_localization_metrics(
                rollout_heatmap,
                boxes,
                image_size=image_size,
                original_width=original_width,
                original_height=original_height,
                heatmap_percentile=heatmap_percentile,
            )

            records.append(
                {
                    "dataset_index": dataset_index,
                    "image_name": image_name,
                    "metadata": metadata,
                    "boxes": boxes,
                    "image_array": image_array,
                    "single_heatmap": single_heatmap,
                    "rollout_heatmap": rollout_heatmap,
                    "single_overlay": single_overlay,
                    "rollout_overlay": rollout_overlay,
                    "true_labels": true_labels,
                    "true_label_text": ", ".join(true_labels) if true_labels else "None",
                    "top_predictions": top_predictions,
                    "top_prediction_text": ", ".join(
                        f"{name}={score:.2f}" for name, score in top_predictions
                    ),
                    "bbox_labels": bbox_labels,
                    "bbox_label_text": ", ".join(bbox_labels) if bbox_labels else "None",
                    "single_metrics": single_metrics,
                    "rollout_metrics": rollout_metrics,
                    "original_width": original_width,
                    "original_height": original_height,
                    "attention_layer_index": layer_index_resolved,
                    "head_reduction": head_reduction,
                    "rollout_head_reduction": rollout_head_reduction,
                }
            )

    return records


if bbox_subset_loader is None:
    attention_records = []
    print("Skipping attention generation because no bbox subset loader is available.")
else:
    attention_records = generate_bbox_attention_records(
        model=model,
        bbox_subset_loader=bbox_subset_loader,
        selected_bbox_samples=selected_bbox_samples,
        selected_bbox_images=selected_bbox_images,
        label_names=label_names,
        device=device,
        image_size=image_size,
        patch_size=patch_size,
        mean=mean,
        std=std,
        layer_index=LAYER_INDEX,
        head_reduction=HEAD_REDUCTION,
        rollout_head_reduction=ROLLOUT_HEAD_REDUCTION,
        rollout_discard_ratio=ROLLOUT_DISCARD_RATIO,
        heatmap_percentile=HEATMAP_PERCENTILE,
        top_k=TOP_K_PREDICTIONS,
    )
    print("Generated attention records:", len(attention_records))
    if attention_records:
        sample_record = attention_records[0]
        print("Example image:", sample_record["image_name"])
        print("BBox labels:", sample_record["bbox_label_text"])
        print("True labels:", sample_record["true_label_text"])
        print("Top predictions:", sample_record["top_prediction_text"])


In [ ]:
def plot_bbox_attention_record(record: dict[str, object]) -> plt.Figure:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(record["image_array"], cmap="gray")
    axes[0].set_title("Original image")

    axes[1].imshow(record["image_array"], cmap="gray")
    axes[1].set_title("Original + bbox")

    axes[2].imshow(record["single_overlay"])
    axes[2].set_title(
        "Single-layer CLS attention
"
        f"Overlap={record['single_metrics']['attention_bbox_overlap']:.2f} | "
        f"IoU={record['single_metrics']['attention_bbox_iou']:.2f}"
    )

    axes[3].imshow(record["rollout_overlay"])
    axes[3].set_title(
        "Attention rollout
"
        f"Overlap={record['rollout_metrics']['attention_bbox_overlap']:.2f} | "
        f"IoU={record['rollout_metrics']['attention_bbox_iou']:.2f}"
    )

    for axis in axes[1:]:
        draw_bboxes_on_axis(
            axis,
            record["boxes"],
            image_size=image_size,
            original_width=record["original_width"],
            original_height=record["original_height"],
        )

    for axis in axes:
        axis.axis("off")

    fig.suptitle(
        f"{record['image_name']} | BBox label(s): {record['bbox_label_text']} | "
        f"True: {record['true_label_text']} | "
        f"Top pred: {record['top_prediction_text']} | "
        f"layer={record['attention_layer_index']} | heads={record['head_reduction']}",
        fontsize=11,
    )
    fig.tight_layout()
    return fig


saved_paths = []
if not attention_records:
    print("No bbox-aware attention records are available to plot.")
else:
    for sample_index, record in enumerate(attention_records):
        fig = plot_bbox_attention_record(record)
        plt.show()
        output_path = FIGURE_OUTPUT_DIR / f"bbox_sample_{sample_index:03d}_{str(record['image_name']).replace('/', '_')}.png"
        fig.savefig(output_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        saved_paths.append(output_path)

    print(f"Saved {len(saved_paths)} bbox attention figures to {FIGURE_OUTPUT_DIR}")


In [ ]:
metrics_rows = []
for index, record in enumerate(attention_records):
    output_path = str(saved_paths[index]) if index < len(saved_paths) else ""
    metrics_rows.append(
        {
            "dataset_index": record["dataset_index"],
            "image_name": record["image_name"],
            "bbox_labels": record["bbox_label_text"],
            "true_labels": record["true_label_text"],
            "top_predictions": record["top_prediction_text"],
            "attention_layer_index": record["attention_layer_index"],
            "head_reduction": record["head_reduction"],
            "rollout_head_reduction": record["rollout_head_reduction"],
            "single_layer_attention_bbox_overlap": record["single_metrics"]["attention_bbox_overlap"],
            "single_layer_attention_bbox_iou": record["single_metrics"]["attention_bbox_iou"],
            "single_layer_heatmap_mass_inside_bbox": record["single_metrics"]["heatmap_mass_inside_bbox"],
            "rollout_attention_bbox_overlap": record["rollout_metrics"]["attention_bbox_overlap"],
            "rollout_attention_bbox_iou": record["rollout_metrics"]["attention_bbox_iou"],
            "rollout_heatmap_mass_inside_bbox": record["rollout_metrics"]["heatmap_mass_inside_bbox"],
            "output_path": output_path,
        }
    )

metrics_df = pd.DataFrame(metrics_rows)
if metrics_df.empty:
    print("No localization metrics were produced because no bbox-aware attention records were generated.")
else:
    metrics_df.to_csv(METRICS_CSV, index=False)
    print("Saved localization metrics CSV to:", METRICS_CSV)
    display(metrics_df)


In [ ]:
print("Interpretability notebook summary")
print("- Split:", SPLIT)
print("- Selected labels:", SELECTED_LABELS)
print("- Max samples per label:", MAX_SAMPLES_PER_LABEL)
print("- Selected bbox sample CSV:", SELECTED_SAMPLES_CSV)
print("- Figure directory:", FIGURE_OUTPUT_DIR)
print("- Metrics CSV:", METRICS_CSV)
print("- Attention maps in this notebook come from bbox-selected samples, not random dataloader batches.")
print("- To change the study set, edit SELECTED_LABELS, MAX_SAMPLES_PER_LABEL, SPLIT, or CHECKPOINT_PATH and rerun the downstream cells.")
